# Pixal3D · Kaggle T4 最小可运行验证

目标只有一个：**在 Kaggle 单张 T4 上把官方 Pixal3D 示例图生成 `output.glb`**。

这不是 Hub Worker 版本，也不做双卡并发。第一阶段固定：

- GPU：只使用 `cuda:0`（T4 单卡 16 GB）
- Python：3.10 独立 `.venv`
- PyTorch：2.6.0 + CUDA 12.4
- Attention：`SDPA`，不安装 flash-attn
- Pixal3D：官方 `TencentARC/Pixal3D` main
- 推理：`--low_vram --resolution 1024`
- T4 CUDA 架构：自动读取；T4 应显示 `7.5 / sm_75`

> Kaggle Notebook 请打开 **Internet**，Accelerator 选择 **GPU T4 x2**。虽然 Kaggle 提供两张卡，本 Notebook 故意只用第 0 张，先验证单卡可运行性。

官方说明 low-VRAM 模式峰值约 10–12 GB；官方同时提示 Hugging Face Demo 的 H-series 预编译依赖可能与其他 GPU 架构不兼容，因此这里优先为当前 GPU 从源码编译 native CUDA 扩展。


## Cell 1 · 环境与 GPU 诊断

先确认 Kaggle 当前的 GPU、驱动、CUDA Toolkit 和磁盘/RAM。这里不修改环境。


In [ ]:
import os, platform, shutil, subprocess, sys
from pathlib import Path

print("Python(kernel):", sys.version)
print("Platform:", platform.platform())
print("Disk:", shutil.disk_usage('/kaggle/working') if Path('/kaggle/working').exists() else shutil.disk_usage('/'))

for cmd in [
    ['nvidia-smi'],
    ['nvcc', '--version'],
    ['bash', '-lc', "free -h"],
]:
    print("\n$", ' '.join(cmd))
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(p.stdout)


## Cell 2 · 建立 Python 3.10 / Torch 2.6 CUDA 12.4 Runtime

Pixal3D 官方 HF 环境提供 Torch 2.6 + cu124 基线。为了避免 Kaggle Notebook 内核 Python 版本变化影响 CUDA wheel/扩展 ABI，这里用 `uv` 建独立 Python 3.10 环境。


In [ ]:
import os, subprocess, sys
from pathlib import Path

ROOT = Path('/kaggle/working/pixal3d-t4')
VENV = ROOT / '.venv'
PY = VENV / 'bin/python'
PIP = [str(PY), '-m', 'pip']
ROOT.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, env=None):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'uv'], check=True)
if not PY.exists():
    run(['uv', 'venv', '--python', '3.10', str(VENV)])

run(PIP + ['install', '-U', 'pip', 'setuptools', 'wheel', 'ninja', 'packaging'])
run(PIP + [
    'install',
    '--index-url', 'https://download.pytorch.org/whl/cu124',
    'torch==2.6.0', 'torchvision==0.21.0'
])

run([str(PY), '-c',
     "import torch; print('torch=',torch.__version__,'cuda=',torch.version.cuda,'available=',torch.cuda.is_available()); "
     "print('gpu=',torch.cuda.get_device_name(0)); print('cc=',torch.cuda.get_device_capability(0))"])


## Cell 3 · Clone Pixal3D / TRELLIS.2 与基础 Python 依赖

TRELLIS.2 只用于提供 O-Voxel 子包；Pixal3D 自身已经包含其模型代码。这里不使用 `requirements-hfdemo.txt` 的 H-series native wheels。


In [ ]:
import os, subprocess
from pathlib import Path

ROOT = Path('/kaggle/working/pixal3d-t4')
VENV = ROOT / '.venv'
PY = VENV / 'bin/python'
PIP = [str(PY), '-m', 'pip']
PIXAL = ROOT / 'Pixal3D'
TRELLIS = ROOT / 'TRELLIS.2'


def run(cmd, cwd=None, env=None):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

if not (PIXAL / '.git').exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/TencentARC/Pixal3D.git', str(PIXAL)])
else:
    run(['git', '-C', str(PIXAL), 'pull', '--ff-only'])

if not (TRELLIS / '.git').exists():
    run(['git', 'clone', '--depth', '1', '--recursive', 'https://github.com/microsoft/TRELLIS.2.git', str(TRELLIS)])

run(PIP + ['install', '-r', str(PIXAL / 'requirements.txt')])
run(PIP + ['install',
    'https://github.com/LDYang694/Storages/releases/download/20260430/utils3d-0.0.2-py3-none-any.whl'
])


## Cell 4 · 为当前 T4 编译 CUDA / Triton native 扩展

这是最可能暴露兼容问题的一格，故意单独放置。

会安装：`natten 0.21.0`、`nvdiffrast`、`nvdiffrec_render`、`CuMesh`、`FlexGEMM`、`o_voxel`。T4 的 compute capability 应为 **7.5**；`TORCH_CUDA_ARCH_LIST` 和 `NATTEN_CUDA_ARCH` 会自动取当前 GPU 的 capability。

如果这一格失败，请保留完整报错；不要继续推理。


In [ ]:
import os, subprocess
from pathlib import Path

ROOT = Path('/kaggle/working/pixal3d-t4')
VENV = ROOT / '.venv'
PY = VENV / 'bin/python'
PIP = [str(PY), '-m', 'pip']
TRELLIS = ROOT / 'TRELLIS.2'
EXT = ROOT / 'extensions'
EXT.mkdir(exist_ok=True)

# 从 venv 的 torch 查询真实 GPU 架构，而不是猜。
cc = subprocess.check_output([
    str(PY), '-c',
    "import torch; a,b=torch.cuda.get_device_capability(0); print(f'{a}.{b}')"
], text=True).strip()
print('CUDA compute capability =', cc)

build_env = os.environ.copy()
build_env['CUDA_VISIBLE_DEVICES'] = '0'
build_env['TORCH_CUDA_ARCH_LIST'] = cc
build_env['NATTEN_CUDA_ARCH'] = cc
build_env['NATTEN_N_WORKERS'] = '2'   # Kaggle RAM 紧张，避免并行编译吃爆内存
build_env['MAX_JOBS'] = '2'
build_env['ATTN_BACKEND'] = 'sdpa'
build_env['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'

# 尽量指向 Kaggle 当前 nvcc 所属 toolkit。
nvcc = subprocess.check_output(['bash','-lc','command -v nvcc'], text=True).strip()
if nvcc:
    build_env['CUDA_HOME'] = str(Path(nvcc).resolve().parent.parent)
print('CUDA_HOME =', build_env.get('CUDA_HOME'))
print('TORCH_CUDA_ARCH_LIST =', build_env['TORCH_CUDA_ARCH_LIST'])


def run(cmd, cwd=None):
    print('\n$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=build_env, check=True)

# 1) NATTEN: Pixal3D 官方固定 0.21.0
run(PIP + ['install', 'natten==0.21.0', '--no-build-isolation', '--no-deps'])

# 2) nvdiffrast
nvd = EXT / 'nvdiffrast'
if not nvd.exists():
    run(['git', 'clone', '--depth', '1', '-b', 'v0.4.0', 'https://github.com/NVlabs/nvdiffrast.git', str(nvd)])
run(PIP + ['install', str(nvd), '--no-build-isolation'])

# 3) nvdiffrec renderutils
ndr = EXT / 'nvdiffrec'
if not ndr.exists():
    run(['git', 'clone', '--depth', '1', '-b', 'renderutils', 'https://github.com/JeffreyXiang/nvdiffrec.git', str(ndr)])
run(PIP + ['install', str(ndr), '--no-build-isolation'])

# 4) CuMesh
cumesh = EXT / 'CuMesh'
if not cumesh.exists():
    run(['git', 'clone', '--depth', '1', '--recursive', 'https://github.com/JeffreyXiang/CuMesh.git', str(cumesh)])
run(PIP + ['install', str(cumesh), '--no-build-isolation'])

# 5) FlexGEMM
flex = EXT / 'FlexGEMM'
if not flex.exists():
    run(['git', 'clone', '--depth', '1', '--recursive', 'https://github.com/JeffreyXiang/FlexGEMM.git', str(flex)])
run(PIP + ['install', str(flex), '--no-build-isolation'])

# 6) O-Voxel from TRELLIS.2
ovo = TRELLIS / 'o-voxel'
if not ovo.exists():
    raise FileNotFoundError(f'Missing {ovo}; TRELLIS.2 clone/submodules may be incomplete')
run(PIP + ['install', str(ovo), '--no-build-isolation'])

print('\nNative extensions installed.')


## Cell 5 · Import / CUDA smoke test

先不下载 24GB 级模型，只检查依赖能否 import，以及 NATTEN 是否真的针对当前 GPU 可用。


In [ ]:
import os, subprocess
from pathlib import Path

ROOT = Path('/kaggle/working/pixal3d-t4')
PY = ROOT / '.venv/bin/python'
PIXAL = ROOT / 'Pixal3D'

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['ATTN_BACKEND'] = 'sdpa'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

test = r"""
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('GPU', torch.cuda.get_device_name(0), 'CC', torch.cuda.get_device_capability(0))
import natten; print('natten', getattr(natten, '__version__', '?'))
import cumesh; print('cumesh OK')
import flex_gemm; print('flex_gemm OK')
import nvdiffrast.torch as dr; print('nvdiffrast OK')
import o_voxel; print('o_voxel OK')
from pixal3d.pipelines import Pixal3DImageTo3DPipeline
print('Pixal3D pipeline import OK')
"""
subprocess.run([str(PY), '-c', test], cwd=PIXAL, env=env, check=True)


## Cell 6 · 下载权重前的资源检查

Pixal3D 会通过 Hugging Face 按需下载模型权重。这里显示当前磁盘、RAM 和 GPU 空闲量。如果你使用私有 HF 缓存/Token，可在 Kaggle Secrets 中设置 `HF_TOKEN`；官方 Pixal3D 仓库本身是公开模型。


In [ ]:
import os, shutil, subprocess
from pathlib import Path

ROOT = Path('/kaggle/working/pixal3d-t4')
HF_HOME = ROOT / 'hf-cache'
HF_HOME.mkdir(exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

print('HF_HOME =', HF_HOME)
print('disk =', shutil.disk_usage('/kaggle/working'))
subprocess.run(['bash','-lc','free -h'], check=False)
subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total,memory.used,memory.free','--format=csv'], check=False)


## Cell 7 · 最小 Pixal3D 推理：官方示例图 → output.glb

核心验证 Cell。强制：

`CUDA_VISIBLE_DEVICES=0` + `ATTN_BACKEND=sdpa` + `--low_vram --resolution 1024`

第一次运行会下载较大的 Hugging Face 权重，因此日志长是正常的。成功标志是 `/kaggle/working/pixal3d-t4/output.glb` 存在且大小 > 0。


In [ ]:
import os, subprocess, time
from pathlib import Path

ROOT = Path('/kaggle/working/pixal3d-t4')
PY = ROOT / '.venv/bin/python'
PIXAL = ROOT / 'Pixal3D'
INPUT = PIXAL / 'assets/images/0_img.png'
OUTPUT = ROOT / 'output.glb'
HF_HOME = ROOT / 'hf-cache'

assert INPUT.exists(), INPUT

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
env['ATTN_BACKEND'] = 'sdpa'
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['HF_HOME'] = str(HF_HOME)
env['TOKENIZERS_PARALLELISM'] = 'false'

cmd = [
    str(PY), 'inference.py',
    '--image', str(INPUT),
    '--output', str(OUTPUT),
    '--low_vram',
    '--resolution', '1024',
    '--seed', '42',
]
print('$', ' '.join(cmd))
t0 = time.time()
subprocess.run(cmd, cwd=PIXAL, env=env, check=True)
print(f'elapsed = {(time.time()-t0)/60:.2f} min')
assert OUTPUT.exists() and OUTPUT.stat().st_size > 0, 'output.glb was not created'
print('SUCCESS:', OUTPUT, f'{OUTPUT.stat().st_size/1024/1024:.2f} MiB')


## Cell 8 · 结果与资源状态

如果上格成功，这一格展示 GLB 路径，并输出最后的 GPU/RAM 状态。Kaggle 右侧 Files 面板也可以直接看到并保存 `output.glb`。


In [ ]:
from pathlib import Path
import subprocess

OUT = Path('/kaggle/working/pixal3d-t4/output.glb')
print('exists =', OUT.exists())
if OUT.exists():
    print('GLB =', OUT)
    print('size = %.2f MiB' % (OUT.stat().st_size / 1024 / 1024))

subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total,memory.used,memory.free','--format=csv'], check=False)
subprocess.run(['bash','-lc','free -h'], check=False)


## 故障定位顺序

- **Cell 2 失败**：Python/Torch/CUDA 基线问题。
- **Cell 4 失败**：T4 `sm_75` native CUDA 扩展兼容问题，这是当前最需要解决的一层。
- **Cell 5 失败**：扩展虽然安装成功，但 ABI / CUDA runtime 不兼容。
- **Cell 7 下载时失败**：Kaggle Internet、HF 下载或磁盘问题。
- **Cell 7 OOM**：记录 OOM 前日志与 `nvidia-smi`；当前已使用官方 low-VRAM + 1024 最保守路径。
- **Cell 7 成功**：下一步再做 Hub Worker 和第二张 T4 的利用方式。
